In [ ]:
import numpy as np
import pandas as pd
import glob
import os  

# ===============================================
# A) DATA LOADING AND PREPARATION (WITH NEW FEATURES)
# ===============================================

from prepare_basic_dataset import FINAL_FEATURES

data_path = "atp_basic_dataset.npz"

# 1. Data Loading
if not os.path.exists(data_path):
    raise FileNotFoundError(f"{data_path} not found!")

data = np.load(data_path, allow_pickle=True) # allow_pickle=True may be needed for string arrays

# 2. Extract Variables
X_train = data['X_train']
Y_train = data['Y_train']
X_val = data['X_val']
Y_val = data['Y_val']
X_test = data['X_test']
Y_test = data['Y_test']
X_max_per_feature = data['X_max']

# --- GET FEATURE NAMES ---
FINAL_FEATURES = data['feature_names'] 

X = X_train
Y = Y_train

# Check
print(f"--- New Data Structure (Anonymized and Extended) ---")
print(f"Feature Names: {FINAL_FEATURES}")
print(f"1. Training Set Size: {X_train.shape[1]} examples")
print(f"2. Validation Set Size: {X_val.shape[1]} examples")
print(f"3. Test Set Size: {X_test.shape[1]} examples")

# ===============================================
# CONFIGURATION: HYPER-PARAMETERS (ADJUSTABLE SECTION)
# ===============================================
N_X = X.shape[0]        # Input Size
N_H = 32                # Hidden Layer Neuron Count (CAN BE CHANGED FOR EXPERIMENTATION!)
N_Y = Y.shape[0]        # Output Size
LEARNING_RATE = 0.05    # Learning Rate (CAN BE CHANGED FOR EXPERIMENTATION!)
EPOCHS = 5000           # Training Epoch Count

print(f"Input Feature Count (N_X): {N_X}")

# ===============================================
# A) HELPER FUNCTIONS
# ===============================================

def sigmoid(Z):
    """Computes the sigmoid activation function."""
    A = 1 / (1 + np.exp(-Z))
    return A

def sigmoid_backward(A):
    """Computes the derivative of the sigmoid function."""
    # This derivative determines how the error propagates back to the hidden layer.
    dZ = A * (1 - A) 
    return dZ

# ===============================================
# B) MODEL CLASS AND INITIALIZATION
# ===============================================

def initialize_parameters(n_x, n_h, n_y):
    """
    Initializes weights and biases using the "Xavier (Glorot)" method.
    This prevents the 'vanishing gradient' problem for tanh activation.
    """

    # Incorrect Method (leads to vanishing gradients):
    # W1 = np.random.randn(n_h, n_x) * 0.01
    # W2 = np.random.randn(n_y, n_h) * 0.01

    # CORRECT METHOD (Xavier):
    # Preserve the variance by dividing by the number of incoming neurons.
    
    W1 = np.random.randn(n_h, n_x) * np.sqrt(1 / n_x)
    b1 = np.zeros((n_h, 1))
    
    W2 = np.random.randn(n_y, n_h) * np.sqrt(1 / n_h)
    b2 = np.zeros((n_y, 1))
    
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    return parameters


def forward_propagation(X, parameters):
    """Input data and calculates the predicted output."""
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # 1. Layer (Hidden Layer)
    # Z1 = W1 * X + b1
    Z1 = np.dot(W1, X) + b1
    A1 = np.tanh(Z1) 
    
    # 2. Layer (Output Layer)
    # Z2 = W2 * A1 + b2
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2) # Prediction (Y_hat)
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# D) COST FUNCTION
def compute_cost(A2, Y):
    """
    Computes Binary Cross-Entropy cost.
    
    A2: Network predictions (Y_hat)
    Y: True Labels
    """
    m = Y.shape[1] # Number of examples (number of columns)
    
    epsilon = 1e-8 # Very small number
    A2_clipped = np.clip(A2, epsilon, 1 - epsilon)
    
    # Compute the two main terms of the Cross-Entropy formula:
    # Term 1: Y * log(A2_clipped)
    log_probs = Y * np.log(A2_clipped)
    
    # Term 2: (1 - Y) * log(1 - A2_clipped)
    log_probs_complement = (1 - Y) * np.log(1 - A2_clipped)
    
    # Total Cost J = -(1/m) * sum of all elements in (term1 + term2) matrix
    cost = - (1 / m) * (np.sum(log_probs + log_probs_complement))
    
    # The cost should be returned as a single number (scalar).
    cost = np.squeeze(cost) 
    
    return cost


def backward_propagation(parameters, cache, X, Y):
    """
    Computes gradients for a 2-layer neural network using backward propagation.

    parameters: Dictionary containing W1, b1, W2, b2.
    cache: Dictionary containing A1 and A2 from forward propagation.
    X: Input data.
    Y: True labels.

    Returns:
        grads: Dictionary containing gradients dW1, db1, dW2, db2.
    """
    # 1. Retrieve Required Variables
    m = X.shape[1]  # Number of examples
    A1 = cache["A1"]
    A2 = cache["A2"]
    
    # Retrieve parameters
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    
    # 2. Output Layer Gradients (Layer 2)
    
    # dZ2 = A2 - Y (Simplified formula derived from Cross-Entropy + Sigmoid combination)
    dZ2 = A2 - Y
    
    # dW2 = (1/m) * dZ2 . A1^T
    dW2 = (1 / m) * np.dot(dZ2, A1.T)
    
    # db2 = mean of dZ2 over the examples
    db2 = np.mean(dZ2, axis=1, keepdims=True)
    
    # 3. Hidden Layer Gradients (Layer 1)
    
    # Backpropagating the error: W2^T . dZ2
    # This distributes the error from W2 back to the neurons in Layer 1.
    dZ1_weighted = np.dot(W2.T, dZ2)
    
    # tanh derivative: 1 - A1^2
    dZ1 = dZ1_weighted * (1 - np.power(A1, 2))
    
    # dW1 = (1/m) * dZ1 . X^T
    dW1 = (1 / m) * np.dot(dZ1, X.T)
    
    # db1 = mean of dZ1 over the examples
    db1 = np.mean(dZ1, axis=1, keepdims=True)
    
    # Collect gradients in a dictionary
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    
    return grads

# F) PARAMETER UPDATE (GRADIENT DESCENT)
def update_parameters(parameters, grads, learning_rate):
    
    # Retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # Retrieve gradients
    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]
    
    # Apply Gradient Descent Rule
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    
    # Save updated parameters in a dictionary
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    
    return parameters


# G) MAIN TRAINING FUNCTION
def train_ann(X, Y, X_val, Y_val, n_h, learning_rate, num_epochs, print_cost=False): # [1] ADDITION: X_val, Y_val arguments
    
    n_x = X.shape[0] # Input size (2)
    n_y = Y.shape[0] # Output size (1)
    
    # 1. INITIALIZE PARAMETERS
    parameters = initialize_parameters(n_x, n_h, n_y)
    
    costs = []
    validation_costs = [] # [2] ADDITION: List to hold validation costs
    
    # 2. TRAINING LOOP
    for i in range(num_epochs):
        
        # 1. FORWARD PROPAGATION (on Training Set)
        A2, cache = forward_propagation(X, parameters)
        
        # 2. COST CALCULATION (Training Cost)
        cost = compute_cost(A2, Y)
        
        # 3. BACKWARD PROPAGATION
        grads = backward_propagation(parameters, cache, X, Y)
        
        # 4. PARAMETER UPDATE
        parameters = update_parameters(parameters, grads, learning_rate)
        
        # [3] ADDITION: VALIDATION COST CALCULATION
        # Using trained parameters on X_val (only forward propagation)
        A2_val, _ = forward_propagation(X_val, parameters) 
        cost_val = compute_cost(A2_val, Y_val)
        
        # Cost tracking and printing
        if i % (EPOCHS / 10 ) == 0:
            costs.append(cost)
            validation_costs.append(cost_val) # [4] ADDITION: Add validation cost to list
            
            if print_cost:
                print(f"Epoch {i} | Training Cost: {cost:.4f} | Validation Cost: {cost_val:.4f}") # Updated
                
    # -------------------------------------------------------------
    # !!! NEW ADDITION: ADD FINAL COST AFTER LOOP ENDS !!!
    # -------------------------------------------------------------
    # Calculate final cost (even if loop ended at num_epochs-1)
    A2_final, _ = forward_propagation(X, parameters)
    cost_final = compute_cost(A2_final, Y)
    A2_val_final, _ = forward_propagation(X_val, parameters) 
    cost_val_final = compute_cost(A2_val_final, Y_val)
    
    costs.append(cost_final)
    validation_costs.append(cost_val_final)
    
    if print_cost:
        # Print the cost at epoch 10000
        print(f"Epoch {num_epochs} | Training Cost: {cost_final:.4f} | Validation Cost: {cost_val_final:.4f}")
    
    return parameters, costs, validation_costs 
                
    return parameters, costs, validation_costs # [5] UPDATE: Also return validation costs
# H) START TRAINING AND VISUALIZE RESULTS

print(f"--- Training ANN ---")
print(f"Input Size (N_X): {N_X} | Hidden Layer (N_H): {N_H} | Example Count (m): {X.shape[1]}")
print(f"Learning Rate: {LEARNING_RATE} | Epoch Count: {EPOCHS}\n")

# Model Training (!!! TRAINING STARTS HERE !!!)
optimized_parameters, costs, validation_costs = train_ann( # Returning 3 values
    X=X_train, 
    Y=Y_train, 
    X_val=X_val, # Validation Set
    Y_val=Y_val, # Validation Labels
    n_h=N_H, 
    learning_rate=LEARNING_RATE, 
    num_epochs=EPOCHS, 
    print_cost=True
)

print(f"\n--- Training Completed ---")
print(f"Initial Training Cost: {costs[0]:.4f}")
print(f"Final Training Cost (Epoch {EPOCHS}): {costs[-1]:.4f}")
print(f"Final Validation Cost: {validation_costs[-1]:.4f}") # Also print validation result

# Visualize Results
epochs_list = np.arange(0, EPOCHS + 1, (EPOCHS / 10)) # X-axis (0, 1000, 2000, ...) is prepared.

# Visualize Results
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))
# Training Cost
plt.plot(epochs_list, costs, label="Training Cost")
# Validation Cost
plt.plot(epochs_list, validation_costs, label="Validation Cost")

plt.title(f"Cost Reduction (LR: {LEARNING_RATE}, N_H: {N_H})")
plt.xlabel("Epoch Count")
plt.ylabel("Cost (J)")
plt.legend()
plt.grid(True)
plt.show()

# J) ACCURACY CALCULATION FUNCTION
def evaluate_accuracy(Y_prediction, Y_true):
    """
    Calculates the model's accuracy score by comparing predictions (Y_prediction)
    with true labels (Y_true).
    """
    # In NumPy, == is used to compare two matrices. 
    # The result is a matrix of (True=1, False=0).
    correct_predictions = (Y_prediction == Y_true)
    
    # Taking the mean directly gives us the ratio of correct predictions (Accuracy).
    accuracy = np.mean(correct_predictions) 
    
    return accuracy


# I) PREDICTION FUNCTION
def predict(parameters, X):
    """
    Makes predictions for input X using trained parameters.
    
    Arguments:
    parameters -- Dictionary of trained W and b parameters
    X -- Input matrix to predict (N, m)
    
    Returns:
    Y_prediction -- NumPy vector of 0 or 1 labels for input X
    """
    
    # Only Forward Propagation
    A2, cache = forward_propagation(X, parameters)
    
    # Decision Threshold: If probability is greater than 0.5, it's 1 (won), otherwise 0 (lost).
    Y_prediction = (A2 > 0.5).astype(int)
    
    return Y_prediction

# K) FINAL PERFORMANCE AND RESULTS BLOCK

# 1. Test Set Prediction (2024 data)
# optimized_parameters: trained W and b values from train_ann
Y_pred_test = predict(optimized_parameters, X_test)

# 2. Accuracy Calculation
# Model's final performance on 2024 data
test_accuracy = evaluate_accuracy(Y_pred_test, Y_test)

print(f"\n--- Model Evaluation ---")
print(f"Training Set Size (1968-2021): {X.shape[1]}")
print(f"Validation Set Size (2022-2023): {X_val.shape[1]}")
print(f"Test Set Size (2024): {X_test.shape[1]}")
print("-" * 35)
print(f"Final TEST Accuracy (2024): {test_accuracy*100:.2f}%")
print("-" * 35)

# ===============================================
# FIRST ADD THIS: I) PREDICTION FUNCTION (UPDATED)
# ===============================================
# Add this version right after or in place of the existing 'predict' function.
# This function returns both the winner (0/1) and the probability (0.0 - 1.0).

def predict_with_proba(parameters, X):
    """
    Returns both the prediction class (0/1) and the raw probability value (A2).
    """
    # Forward Propagation
    A2, _ = forward_propagation(X, parameters)
    
    # Decision: 1 if greater than 0.5
    Y_prediction = (A2 > 0.5).astype(int)
    
    return Y_prediction, A2

# ===============================================
# K) FINAL PERFORMANCE AND SELECTIVE ACCURACY ANALYSIS (ADAPTED FOR 1 LAYER)
# ===============================================

import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

# 1. Test Set Prediction (with Probabilities)
if 'optimized_parameters' in locals() and optimized_parameters:
    
    # Get predictions and probabilities using the new function
    Y_pred_test, A_test_proba = predict_with_proba(optimized_parameters, X_test)

    # 2. Overall Accuracy Calculation (ALL MATCHES)
    test_accuracy = evaluate_accuracy(Y_pred_test, Y_test)

    print(f"\n--- Model Evaluation (Overall Result) ---")
    print(f"Test Set (2024) Sample Count: {X_test.shape[1]}")
    print("-" * 35)
    print(f"Final TEST Accuracy (All Matches): {test_accuracy*100:.2f}%")
    print("-" * 35)

    # 3. Create DataFrame for Detailed Match Analysis
    true_labels = Y_test.flatten()
    pred_labels = Y_pred_test.flatten()
    p1_win_probabilities = A_test_proba.flatten() 

    df_results = pd.DataFrame({
        'Actual Winner': np.where(true_labels == 1, 'P1', 'P2'),
        'Predicted': np.where(pred_labels == 1, 'P1', 'P2'),
        'P1 Winning Probability (%)': p1_win_probabilities * 100,
        'Result': np.where(true_labels == pred_labels, 'CORRECT', 'WRONG')
    })
    
    # Calculate confidence score (the further from 50%, the more confident)
    df_results['Prediction Confidence (%)'] = np.where(
        df_results['Predicted'] == 'P1', 
        df_results['P1 Winning Probability (%)'], 
        100.0 - df_results['P1 Winning Probability (%)']
    )
    
    # --- 4. SELECTIVE ACCURACY (CONFIDENCE THRESHOLDING) ---
    print("\n--- Selective Accuracy (Confidence Thresholding) ---")
    
    # Define threshold (e.g., 10 point margin = 60% and above or 40% and below)
    # You can adjust here how "confident" matches you want to look at.
    CONFIDENCE_MARGIN = 10  
    
    CONFIDENCE_THRESHOLD_UPPER = 50 + CONFIDENCE_MARGIN  # E.g., 60.0
    CONFIDENCE_THRESHOLD_LOWER = 50 - CONFIDENCE_MARGIN  # E.g., 40.0

    print(f"Only looking at matches where model predicts P1 > %{CONFIDENCE_THRESHOLD_UPPER} or P1 < %{CONFIDENCE_THRESHOLD_LOWER}...")

    # Filter only "confident" predictions
    confident_predictions_df = df_results[
        (df_results['P1 Winning Probability (%)'] > CONFIDENCE_THRESHOLD_UPPER) |
        (df_results['P1 Winning Probability (%)'] < CONFIDENCE_THRESHOLD_LOWER)
    ]

    total_matches = len(df_results)
    confident_matches = len(confident_predictions_df)
    
    if confident_matches > 0:
        # Calculate accuracy only for these 'confident' matches
        selective_accuracy = (confident_predictions_df['Result'] == 'CORRECT').mean()
        
        print(f"\n   -> Model was confident in {confident_matches} out of {total_matches} matches (%{100*confident_matches/total_matches:.1f}).")
        print(f"   -> NEW accuracy for these 'confident' matches: {selective_accuracy*100:.2f}%")
        print("-" * 35)
    else:
        print("   -> Model was not confident in any match at this threshold.")
        print("-" * 35)
        
    
    # --- 5. VISUALIZATION AND SAVING ---
    
    # 1) Prepare folder
    out_dir = Path('./confidence_imgs_1layer') # Changed folder name to avoid confusion
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2) File naming logic (auto-incrementing number)
    base = 'confidence_dist_1layer'
    numbered_regex = re.compile(rf'^{re.escape(base)}(\d+)\.png$')

    max_idx = 0
    bare_exists = (out_dir / f'{base}.png').exists()

    if bare_exists:
        max_idx = 1

    for p in out_dir.glob(f'{base}[0-9]*.png'):
        m = numbered_regex.match(p.name)
        if m:
            idx = int(m.group(1))
            if idx > max_idx:
                max_idx = idx

    next_idx = max_idx + 1 if (bare_exists or max_idx > 0) else 1
    out_path = out_dir / f'{base}{next_idx}.png'


    # Plot Graph
    print("\nConfidence Distribution (Correct vs Wrong Distribution) is created...")
    plt.figure(figsize=(12, 7))
    bins_range = np.arange(50, 101, 5) # Bar range
    
    # Histograms
    sns.histplot(df_results[df_results['Result'] == 'CORRECT']['Prediction Confidence (%)'], bins=bins_range, kde=False, color='green', alpha=0.6, label='Correct')
    sns.histplot(df_results[df_results['Result'] == 'WRONG']['Prediction Confidence (%)'], bins=bins_range, kde=False, color='red', alpha=0.6, label='Wrong')
    
    plt.title('1-Layer Model: Confidence Distribution (Correct vs Wrong)')
    plt.xlabel('Model Confidence (%)')
    plt.ylabel('Match Count')
    plt.legend()
    
    # Info Box (Updated for 1-Layer Model Variables)
    ax = plt.gca()
    ax.text(
        0.99, 0.99,
        f"Data: {data_path}\n"
        f"Model: 1-Layer NN (Tanh)\n"
        f"Hidden Neurons={N_H}\n"
        f"LR={LEARNING_RATE}\n"
        f"Epochs={EPOCHS}",
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', fc='white', ec='#dddddd', alpha=0.85)
    )
    
    print(f'Confidence Distribution (Correct vs Wrong) is saved: {out_path}')
    plt.grid(True, alpha=0.3)
    plt.savefig(out_path)
    plt.show()

else:
    print("\nPlease train the model first (optimized_parameters not found).")